In [18]:
import random
from itertools import product

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

## Problem 1

### Generating Random Boolean Functions
The Deutsch–Jozsa algorithm is designed to work with functions that accept a fixed number of Boolean inputs and return a single Boolean output. Each function is guaranteed to be either constant (always returns False or always returns True) or balanced (returns True for exactly half of the possible input combinations). Write a Python function random_constant_balanced that returns a randomly chosen function from the set of constant or balanced functions taking four Boolean arguments as inputs.

Concept notes:
- A constant function returns the same output for every input.
- A balanced function returns True for exactly half of all inputs and False for the other half.
- With 4 input bits, there are 16 possible inputs, so a balanced function has 8 True outputs and 8 False outputs.

In [19]:
def random_constant_balanced():
    """
    Return a random 4-input Boolean function f(a, b, c, d) that is promised to be
    either constant or balanced.
    """
    # Build the full 4-bit input space: 2^4 = 16 tuples.
    inputs = list(product([False, True], repeat=4))

    # Randomly choose which promised family to sample from.
    if random.choice(["constant", "balanced"]) == "constant":
        value = random.choice([False, True])

        # Constant function ignores inputs and returns one fixed value.
        def f(a, b, c, d):
            return value

    else:
        # Balanced function is True on exactly half the input space.
        true_set = set(random.sample(inputs, k=len(inputs) // 2))

        def f(a, b, c, d):
            return (a, b, c, d) in true_set

    return f


# Quick sanity check: evaluate one random promised function.
f = random_constant_balanced()
print(f(False, False, False, False))

True


## Problem 2

### Classical Testing for Function Type
Deutsch's algorithm is designed to demonstrate a potential advantage of quantum computing over classical computation. To understand this advantage, we must first understand the classical cost of solving the underlying problem. Write a Python function determine_constant_balanced that takes as input a function f, as defined in Problem 1. The function should analyze f and return the string "constant" or "balanced" depending on whether the function is constant or balanced. Write a brief note on the efficiency of your solution. What is the maximum number of times you need to call f to be 100% certain whether it is constant or balanced?

Concept notes:
- Under the promise, f is only one of two types: constant or balanced.
- The algorithm can stop early once it has enough evidence to reject one type.
- Worst-case classical certainty for a 4-bit promised function is 9 evaluations.

In [20]:
def determine_constant_balanced(f):
    """
    Classify a promised 4-input Boolean function as "constant" or "balanced".
    """
    true_count = 0
    false_count = 0

    # Evaluate the function over the full input space, stopping as soon as possible.
    for a, b, c, d in product([False, True], repeat=4):
        result = bool(f(a, b, c, d))

        if result:
            true_count += 1
        else:
            false_count += 1

        # A balanced 4-bit promised function cannot exceed 8 of one output value.
        if true_count > 8 or false_count > 8:
            return "constant"

        # Seeing both True and False immediately rules out constant.
        if true_count > 0 and false_count > 0:
            return "balanced"

    # If every observed output is identical, the function is constant.
    return "constant"


# Example usage and worst-case query statement.
f = random_constant_balanced()
kind = determine_constant_balanced(f)
print(f"Function type: {kind}")
print("Worst-case classical calls needed for certainty: 9")

Function type: constant
Worst-case classical calls needed for certainty: 9


## Problem 3

### Quantum Oracles
Deutsch's algorithm is the simplest example of a quantum algorithm using superposition to determine a global property of a function with a single evaluation. In the single-input case, there are four possible Boolean functions. Using Qiskit, create the appropriate quantum oracles for each of the possible single-Boolean-input functions used in Deutsch's algorithm. Demonstrate their use and explain how each oracle implements its corresponding function.

Concept notes:
- A Deutsch oracle implements the reversible map |x,y> -> |x, y xor f(x)|.
- Reversibility is required for valid quantum operations.
- Different gate patterns represent the four possible single-bit Boolean functions.

In [21]:
"""
Quantum oracles for Deutsch's algorithm:
- f(x)=0: identity
- f(x)=1: X on target
- f(x)=x: CNOT
- f(x)=not x: X, CNOT, X
"""


def oracle_f0():
    """f(x)=0: |x, y> -> |x, y>"""
    qc = QuantumCircuit(2, name="Uf0")
    # Identity operation: y is unchanged for both x=0 and x=1.
    return qc


def oracle_f1():
    """f(x)=1: |x, y> -> |x, y xor 1>"""
    qc = QuantumCircuit(2, name="Uf1")
    # Always flip target qubit because f(x)=1 for all x.
    qc.x(1)
    return qc


def oracle_fx():
    """f(x)=x: |x, y> -> |x, y xor x>"""
    qc = QuantumCircuit(2, name="Ufx")
    # CNOT flips y iff control x is 1.
    qc.cx(0, 1)
    return qc


def oracle_fnotx():
    """f(x)=not x: |x, y> -> |x, y xor (not x)>"""
    qc = QuantumCircuit(2, name="Ufnotx")
    # Convert control-on-0 into control-on-1 with X gates around CNOT.
    qc.x(0)
    qc.cx(0, 1)
    qc.x(0)
    return qc


# Build and display all four possible single-bit Deutsch oracles.
oracles = {
    "f(x)=0": oracle_f0(),
    "f(x)=1": oracle_f1(),
    "f(x)=x": oracle_fx(),
    "f(x)=not x": oracle_fnotx(),
}

for name, oracle in oracles.items():
    print(f"\n{name}")
    print(oracle.draw(output="text"))


f(x)=0
     
q_0: 
     
q_1: 
     

f(x)=1
          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

f(x)=x
          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

f(x)=not x
     ┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     └───┘┌─┴─┐└───┘
q_1: ─────┤ X ├─────
          └───┘     


## Problem 4

### Deutsch's Algorithm with Qiskit
Use Qiskit to design a quantum circuit that solves Deutsch's problem for a function with a single Boolean input. Implement the necessary circuit and demonstrate its use with each of the quantum oracles from Problem 3. Describe how the interference pattern produced by the circuit allows you to determine whether the function is constant or balanced using only one query to the oracle.

Concept notes:
- Initialize the ancilla in |1> and apply Hadamard so phase kickback can encode f(x) into relative phase.
- Apply Hadamard to the input qubit so the circuit evaluates x=0 and x=1 branches in one oracle query.
- The oracle U_f preserves reversibility via |x,y> -> |x, y xor f(x)| while embedding function information.
- A final Hadamard on the input qubit causes interference: x=0 indicates constant, x=1 indicates balanced (under the promise).

In [22]:
# Ensure Problem 3 was run, because this cell reuses the oracle dictionary.
if "oracles" not in globals():
    raise NameError("Run Problem 3 first so the 'oracles' dictionary is available.")


def deutsch_circuit(oracle: QuantumCircuit) -> QuantumCircuit:
    """
    Build Deutsch's algorithm circuit for a supplied 2-qubit oracle U_f.
    Qubit 0: input register x
    Qubit 1: ancilla y
    """
    qc = QuantumCircuit(2)

    # Prepare |x,y> = |0,1>. The ancilla in |1> is required for phase kickback.
    qc.x(1)

    # Create superposition so one oracle call processes both x=0 and x=1 branches.
    qc.h(0)
    qc.h(1)

    # Single black-box query.
    qc.compose(oracle, inplace=True)

    # Interference step: maps phase information back onto the input qubit.
    qc.h(0)

    return qc


def classify_with_deutsch(oracle: QuantumCircuit) -> tuple[str, list[float]]:
    """
    Classify oracle as constant/balanced from the x-qubit distribution.
    Returns (label, [P(x=0), P(x=1)]).
    """
    qc = deutsch_circuit(oracle)

    # Use exact statevector probabilities to avoid sampling noise.
    probs_x = Statevector.from_instruction(qc).probabilities([0])

    # Deutsch decision rule:
    # x=0 with probability 1 -> constant, x=1 with probability 1 -> balanced.
    label = "constant" if probs_x[0] > 0.999 else "balanced"
    return label, probs_x


print("Deutsch algorithm results (single oracle query):")
for name, oracle in oracles.items():
    label, probs_x = classify_with_deutsch(oracle)
    print(f"{name:10s} -> {label:8s} | P(x=0)={probs_x[0]:.3f}, P(x=1)={probs_x[1]:.3f}")

Deutsch algorithm results (single oracle query):
f(x)=0     -> constant | P(x=0)=1.000, P(x=1)=0.000
f(x)=1     -> constant | P(x=0)=1.000, P(x=1)=0.000
f(x)=x     -> balanced | P(x=0)=0.000, P(x=1)=1.000
f(x)=not x -> balanced | P(x=0)=0.000, P(x=1)=1.000


## Problem 5

### Scaling to the Deutsch–Jozsa Algorithm
The Deutsch–Jozsa algorithm generalizes Deutsch's approach to functions with several input bits. Use Qiskit to create a quantum circuit that can handle the four-bit functions generated in Problem 1. Explain how the classical function is encoded as a quantum oracle, and demonstrate the use of your circuit on both of the constant functions and any two balanced functions of your choosing. Show that the circuit correctly identifies the type of each function.

Concept notes:
- The classical function must be turned into a reversible oracle before it can be used in a quantum circuit.
- The oracle implements |x,y> -> |x, y xor f(x)|, so the input register is preserved and the ancilla stores the function output by XOR.
- Hadamard gates place the input register into uniform superposition, which is what gives the algorithm its parallelism.
- After the oracle, another Hadamard layer makes the constant case interfere to |0000> while balanced functions cancel out that outcome.
- For a promised function, measuring all zeroes means constant; any other bitstring means balanced.

In [23]:
# Turn a classical Boolean function into a reversible Deutsch-Jozsa oracle.
# The oracle must preserve x and compute y xor f(x) on the ancilla.
def build_dj_oracle(f, n=4):
    """
    Build an (n+1)-qubit oracle U_f for a classical promised Boolean function.
    The last qubit is the target ancilla.
    """
    uf = QuantumCircuit(n + 1, name="Uf")

    # Iterate through every possible input bitstring.
    # For each x where f(x)=1, flip the target qubit controlled on that exact x.
    for bits in product([0, 1], repeat=n):
        if f(*map(bool, bits)):
            # Convert 0-controls into 1-controls by temporarily inverting those qubits.
            for index, bit in enumerate(bits):
                if bit == 0:
                    uf.x(index)

            # Multi-controlled X applies the XOR with f(x) on the ancilla.
            uf.mcx(list(range(n)), n)

            # Undo the temporary inversions so the oracle remains reversible.
            for index, bit in enumerate(bits):
                if bit == 0:
                    uf.x(index)

    return uf


# Build the Deutsch-Jozsa circuit for a 4-bit promised function.
def deutsch_jozsa_circuit(f, n=4):
    """
    Create the Deutsch-Jozsa circuit for a promised n-bit function.
    Returns a circuit that can be analyzed with a statevector.
    """
    qc = QuantumCircuit(n + 1)

    # Start the ancilla in |1>. After the Hadamard, it becomes |->,
    # which is what enables phase kickback.
    qc.x(n)

    # Put every qubit into superposition so all inputs are queried in one call.
    for qubit in range(n + 1):
        qc.h(qubit)

    # Insert the reversible oracle corresponding to the classical function.
    qc.compose(build_dj_oracle(f, n=n), inplace=True)

    # Apply Hadamards again on the input register only.
    # This is the interference step that separates constant from balanced cases.
    for qubit in range(n):
        qc.h(qubit)

    return qc


# Classify the promised function from the input-register probabilities.
def classify_deutsch_jozsa(f, n=4):
    """
    Return "constant" or "balanced" using the Deutsch-Jozsa circuit.
    """
    qc = deutsch_jozsa_circuit(f, n=n)

    # Read only the input register probabilities.
    # If the function is constant, the all-zero state appears with probability 1.
    probs = Statevector.from_instruction(qc).probabilities(qargs=list(range(n)))
    return "constant" if probs[0] > 0.999 else "balanced"


# Two constant functions.
def f_const_false(a, b, c, d):
    # Always returns False, so the oracle should act like identity.
    return False


def f_const_true(a, b, c, d):
    # Always returns True, so the oracle flips the ancilla for every input.
    return True


# Two balanced functions.
def f_balanced_parity(a, b, c, d):
    # True when an odd number of inputs are True.
    return a ^ b ^ c ^ d


def f_balanced_first_bit(a, b, c, d):
    # True for exactly half the input space because it depends only on the first bit.
    return a


# Run a small demo so the notebook shows the expected classifications.
tests = {
    "const_false": f_const_false,
    "const_true": f_const_true,
    "balanced_parity": f_balanced_parity,
    "balanced_first_bit": f_balanced_first_bit,
}

for name, fn in tests.items():
    print(f"{name:18s} -> {classify_deutsch_jozsa(fn, n=4)}")

const_false        -> constant
const_true         -> constant
balanced_parity    -> balanced
balanced_first_bit -> balanced
